# Quickstart

This notebook walks through the basic `openindexmaps-py` workflow:

- create one or more sheet features
- assemble them into an `OpenIndexMap`
- validate the output and compute a bounding box
- write GeoJSON to disk and read it back
- generate GeoBlacklight/Aardvark metadata from the collection


In [ ]:
import json
from pathlib import Path

from openindexmaps_py.metadata import GeoBlacklight_Metadata
from openindexmaps_py.oimpy import MapSheet, OpenIndexMap

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent

FILES_DIR = PROJECT_ROOT / "notebooks" / "files"
SCHEMA_PATH = PROJECT_ROOT / "schemas" / "1.0.0.schema.json"
FIXTURE_PATH = PROJECT_ROOT / "tests" / "fixture" / "f0140_OIM.geojson"

FILES_DIR.mkdir(exist_ok=True)
PROJECT_ROOT

## 1. Create sheet features

`MapSheet` builds a polygon from `west`, `east`, `south`, and `north`, then stores the rest of the fields as GeoJSON properties.

In [ ]:
sheet_1 = MapSheet(
    {
        "label": "QS-001",
        "title": "Quickstart Demo Sheet 1",
        "datePub": "1950",
        "publisher": "OpenIndexMaps Demo",
        "scale": "1:24000",
        "west": -89.5,
        "east": -89.25,
        "south": 43.0,
        "north": 43.25,
        "available": True,
    }
)

sheet_1.__geo_interface__

In [ ]:
sheet_2 = MapSheet(
    {
        "label": "QS-002",
        "title": "Quickstart Demo Sheet 2",
        "datePub": "1962",
        "publisher": "OpenIndexMaps Demo",
        "scale": "1:24000",
        "west": -89.25,
        "east": -89.0,
        "south": 43.0,
        "north": 43.25,
        "available": False,
    }
)

[sheet_1.label, sheet_2.label]

## 2. Assemble an OpenIndexMap

`OpenIndexMap` is a GeoJSON `FeatureCollection` wrapper around the sheet features.

In [ ]:
quickstart_oim = OpenIndexMap([sheet_1, sheet_2])

{
    "type": quickstart_oim.__geo_interface__["type"],
    "feature_count": len(quickstart_oim.__geo_interface__["features"]),
}

In [ ]:
quickstart_oim.__geo_interface__

## 3. Validate and inspect spatial extent

The collection can be validated against the repository schema and summarized with a computed bounding box.

In [ ]:
{
    "is_valid": quickstart_oim.is_valid(str(SCHEMA_PATH)),
    "bbox": quickstart_oim.compute_bbox(),
}

## 4. Write GeoJSON to disk and read it back

This is the simplest local round-trip if you want to inspect generated output.

In [ ]:
output_path = FILES_DIR / "quickstart_oim.geojson"

with output_path.open("w") as f:
    json.dump(quickstart_oim.__geo_interface__, f, indent=2)

output_path

In [ ]:
reloaded_oim = OpenIndexMap.from_file(str(output_path))

{
    "feature_count": len(reloaded_oim.__geo_interface__["features"]),
    "bbox": reloaded_oim.compute_bbox(),
}

## 5. Load an existing fixture from the repository

The package can also rebuild an `OpenIndexMap` from an existing GeoJSON file.

In [ ]:
fixture_oim = OpenIndexMap.from_file(str(FIXTURE_PATH))

{
    "fixture_path": str(FIXTURE_PATH.relative_to(PROJECT_ROOT)),
    "feature_count": len(fixture_oim.__geo_interface__["features"]),
    "bbox": fixture_oim.compute_bbox(),
}

## 6. Generate GeoBlacklight metadata

`GeoBlacklight_Metadata` can derive index years and an envelope from an `OpenIndexMap`.

In [ ]:
metadata = GeoBlacklight_Metadata()
metadata.set_attribute("id", "quickstart-demo")
metadata.set_attribute("dct_title_s", "OpenIndexMap: Quickstart Demo")
metadata.set_attribute("gbl_resourceClass_sm", ["Maps"])
metadata.set_attribute("dct_accessRights_s", "Public")
metadata.compute_details_from_oim(quickstart_oim)
metadata.set_references("https://example.org/quickstart_oim.geojson")

metadata.metadata

In [ ]:
metadata_path = FILES_DIR / "quickstart_metadata.json"
metadata.generate_metadata_file(metadata_path)
metadata_path

## Next steps

After this notebook, the next useful paths are:

- experiment with CLI workflows in `docs/04-cli-workflows.md`
- inspect and transform repository fixture files under `tests/fixture/`
- continue the SQLite experiment separately in `sqlite.ipynb`
